# Bundle adjustment of a multi-camera rig

This example walks through the full `deeperfly` workflow:

1. **Load** a 7-camera rig from a TOML config (`cameras.toml`). This is the
   camera-only example file; a full run config (written by `deeperfly init`)
   gathers the same `[cameras]` / `[bundle_adjustment]` sections alongside
   `[inputs]`, `[pipeline]` and `[skeleton]`. `CameraGroup.from_config` reads
   only the cameras, so either file works here.
2. **Synthesize** ground-truth 2D observations of a random 3D point cloud.
3. **Perturb** the camera poses to get a noisy initial guess.
4. **Bundle-adjust** the rig back onto the observations (intrinsics held fixed).
5. **Align** the gauge-free solution to ground truth and compare.

The rig is a microscope-like setup: seven cameras orbiting the world origin on a
circle, all looking inward.

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import expm

from deeperfly import CameraGroup, bundle_adjust, geometry as geom

CONFIG = Path("cameras.toml")  # run this notebook from the examples/ directory

## 1. Load the rig

`CameraGroup.from_config` resolves each camera's extrinsics from the orbit spec
(`look_at` / `azimuth_deg` / `distance`) into canonical `(rvec, tvec)` pairs.

In [ ]:
group = CameraGroup.from_config(CONFIG)
print("cameras:", group.names)
print("intrinsics [fx, fy, cx, cy]:", group.intrs[0])

## 2. Visualize the rig

Each camera is drawn as an RGB triad (x=red, y=green, z=blue/optical axis) at its
world center `-R.T @ t`.

In [ ]:
def plot_camera(rmat, tvec, ax, length=None, **kwargs):
    """Draw a camera as an RGB axis triad at its world center."""
    center = -rmat.T @ tvec
    if length is None:
        length = np.linalg.norm(tvec) * 0.2
    for axis, color in zip(rmat, "rgb"):
        ax.plot(*zip(center, center + axis * length), c=color, **kwargs)


def plot_rig(rvecs, tvecs, ax, **kwargs):
    for rmat, tvec in zip(np.asarray(geom.rvec_to_rmat(rvecs)), tvecs):
        plot_camera(rmat, tvec, ax, **kwargs)


fig, ax = plt.subplots(subplot_kw={"projection": "3d"})
plot_rig(group.rvecs, group.tvecs, ax)
ax.scatter(0, 0, 0, c="k", marker="*", s=80)  # look-at target
ax.set_aspect("equal")
ax.set_title("ground-truth rig")
plt.show()

## 3. Synthesize observations

Sample a random 3D cloud near the origin and project it through every camera to
get the 2D observations we will fit against.

In [ ]:
rng = np.random.default_rng(0)
pts3d_gt = rng.uniform(-0.5, 0.5, size=(100, 3))
pts2d = np.asarray(group.project(pts3d_gt))  # (V, N, 2)
print("observations:", pts2d.shape)

## 4. Perturb the cameras

Bundle adjustment starts from a noisy guess. We rotate each camera by a small
random rotation and jitter its translation.

In [ ]:
def small_rotation(sigma, seed):
    """Random rotation matrix near identity (axis-angle std `sigma`)."""
    o = np.random.default_rng(seed).normal(scale=sigma, size=3)
    return expm(np.array([[0, -o[2], o[1]], [o[2], 0, -o[0]], [-o[1], o[0], 0]]))


rmats0 = np.array(
    [
        small_rotation(0.05, i) @ R
        for i, R in enumerate(np.asarray(geom.rvec_to_rmat(group.rvecs)))
    ]
)
rvecs0 = np.asarray(geom.rmat_to_rvec(rmats0))
tvecs0 = group.tvecs + np.random.default_rng(0).normal(
    scale=5.0, size=group.tvecs.shape
)
cams0 = CameraGroup.from_arrays(group.names, rvecs0, tvecs0, group.intrs, group.dists)

fig, ax = plt.subplots(subplot_kw={"projection": "3d"})
plot_rig(group.rvecs, group.tvecs, ax, alpha=0.3)
plot_rig(cams0.rvecs, cams0.tvecs, ax)
ax.set_aspect("equal")
ax.set_title("ground truth (faded) vs. perturbed initial guess")
plt.show()

## 5. Bundle-adjust

`bundle_adjust` triangulates an initial point cloud from the (perturbed) cameras,
then jointly refines camera poses and 3D points to minimize reprojection error.
We hold all intrinsics fixed via `fixed=["*.intr"]`; the gauge (overall world
pose & scale) is left free, so the cost — not the raw parameters — should go to
zero.

In [ ]:
res, cams_opt, pts3d_opt = bundle_adjust(cams0, pts2d, fixed=["*.intr"], max_nfev=2000)

reproj = np.asarray(cams_opt.project(pts3d_opt)) - pts2d
rmse = np.sqrt(np.nanmean(reproj**2))
print(f"converged: cost={res.cost:.3e}  nfev={res.nfev}  status={res.status}")
print(f"reprojection RMSE: {rmse:.3e} px")

## 6. Align to ground truth and compare

Reprojection error is invariant to a global **similarity** transform of the
scene — rotating, translating *and rescaling* all cameras and points together
leaves every image point unchanged. So minimizing reprojection alone fixes the
geometry only up to this 7-DOF *gauge freedom*; the near-zero RMSE above is the
meaningful convergence check, not the raw parameter values. (To pin the gauge
outright, anchor a camera and fix a distance via the `fixed`/`shared` config —
see `cameras.toml`.)

To *compare* the recovered geometry with ground truth we first remove the gauge
with a closed-form similarity alignment (Umeyama, 1991) of the camera centers,
then apply the same transform to the cameras and points.

In [ ]:
def similarity_align(src, dst):
    """Least-squares similarity (scale s, rotation R, translation t) with
    ``dst ~= s * (R @ src) + t`` (Umeyama 1991)."""
    mu_s, mu_d = src.mean(0), dst.mean(0)
    src_c, dst_c = src - mu_s, dst - mu_d
    cov = dst_c.T @ src_c / len(src)
    U, sigma, Vt = np.linalg.svd(cov)
    signs = np.ones(3)
    signs[-1] = np.sign(np.linalg.det(U @ Vt))  # keep a proper rotation
    R = U @ np.diag(signs) @ Vt
    s = (sigma * signs).sum() / (src_c**2).sum() * len(src)
    return s, R, mu_d - s * R @ mu_s


def camera_centers(rvecs, tvecs):
    rmats = np.asarray(geom.rvec_to_rmat(rvecs))
    return -np.einsum("oij,oi->oj", rmats, tvecs)


centers_gt = camera_centers(group.rvecs, group.tvecs)
s, R, t = similarity_align(camera_centers(cams_opt.rvecs, cams_opt.tvecs), centers_gt)

# Apply X' = s R X + t to the world: cameras transform as R_i' = R_i R^T,
# t_i' = s t_i - R_i' t (projection is scale-invariant), points as X' = s R X + t.
rmats_aligned = np.asarray(geom.rvec_to_rmat(cams_opt.rvecs)) @ R.T
tvecs_aligned = s * cams_opt.tvecs - np.einsum("oij,j->oi", rmats_aligned, t)
rvecs_aligned = np.asarray(geom.rmat_to_rvec(rmats_aligned))
pts3d_aligned = pts3d_opt @ (s * R).T + t

print(f"recovered scale: {s:.4f}")
print(
    "max camera-center error:",
    np.max(
        np.linalg.norm(
            camera_centers(rvecs_aligned, tvecs_aligned) - centers_gt, axis=-1
        )
    ),
)
print("max point error:", np.max(np.linalg.norm(pts3d_aligned - pts3d_gt, axis=-1)))

fig, ax = plt.subplots(subplot_kw={"projection": "3d"})
plot_rig(group.rvecs, group.tvecs, ax, alpha=0.3)
plot_rig(rvecs_aligned, tvecs_aligned, ax)
ax.set_aspect("equal")
ax.view_init(elev=90, azim=0)
ax.set_title("ground truth (faded) vs. recovered (similarity-aligned)")
plt.show()